# TP1 — Resultados — Fases 9 y 10

**73.69 Large Language Models — ITBA, 2026** · Predicción de *Buy Through Rate*

Este notebook es el entregable de la **Fase 9** (estudio de ablación) y la **Fase 10**
(evaluación final e interpretabilidad) del [plan](../plan.md).

## Protocolo

| | |
|---|---|
| Ablaciones | 17 configuraciones × **5 semillas × 5 folds** = 425 entrenamientos |
| Regla | cada fila **declara** qué claves cambia y un test verifica la declaración |
| Partición | CV agrupada por `query_id` sobre `dev`. **El test no se toca en la Fase 9** |
| Selección | la configuración final se elige solo con CV, antes de mirar el test |
| Test | se evalúa **una única vez**, en la Fase 10 |

Con ~200 positivos por fold de validación, diferencias de PR-AUC de pocos puntos son
indistinguibles del ruido. Por eso cada fila se reporta con su desvío y cada comparación
contra la base incluye el error estándar de la diferencia: sin eso, una tabla de ablación
no permite concluir nada.

## Índice

| § | Contenido |
|---|---|
| 1 | La tabla de ablación completa |
| 2 | Comparaciones contra la base, con error estándar |
| 3 | El hallazgo central: el marcador de reputación |
| 4 | Arquitectura: las cuatro familias, con el ratio del paper |
| 5 | Tokenizador: el vocabulario que fragmenta el marcador |
| 6 | La fuga (`cart`) |
| 7 | Evaluación final sobre test |
| 8 | Calibración |
| 9 | Curvas ROC y PR contra los baselines |
| 10 | Diagnóstico de over/underfitting |
| 11 | Mapas de atención |
| 12 | Métricas de ranking |

---
## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluate import format_summary, load_results, summarize

FIGDIR = ROOT / "report" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})

AZUL, GRIS, ROJO, VERDE, NARANJA = "#2b6cb0", "#a0aec0", "#c53030", "#2f855a", "#dd6b20"


def guardar(fig, nombre):
    ruta = FIGDIR / f"{nombre}.png"
    fig.savefig(ruta)
    print(f"guardada: {ruta.relative_to(ROOT)}")


resultados = load_results()
ablaciones = resultados.loc[resultados["config"] == "ablacion"]
baselines = resultados.loc[resultados["config"] == "baseline"]
print(f"{len(ablaciones)} corridas de ablacion "
      f"({ablaciones['run_name'].nunique()} configuraciones x "
      f"{ablaciones['seed'].nunique()} semillas x {ablaciones['fold'].nunique()} folds)")

---
## 1. La tabla de ablación completa

In [ ]:
# Cada configuracion con su descripcion, en el orden en que las define la grilla.
from src.ablations import GRILLA

ORDEN = [c.name for c, _ in GRILLA]
NOTAS = {c.name: n for c, n in GRILLA}

resumen = summarize(ablaciones).set_index("run_name").reindex(ORDEN)
tabla = format_summary(resumen.reset_index())
tabla.insert(1, "que cambia", [NOTAS[n].split(".")[0] for n in tabla["run_name"]])
tabla

---
## 2. Comparaciones contra la base

Una diferencia solo significa algo si supera al ruido. Para cada configuración se reporta
la diferencia de PR-AUC contra la base y el **error estándar de la diferencia**,
`sqrt(sd_a²/n_a + sd_b²/n_b)` con n = 25 corridas cada una. El veredicto usa la regla
habitual de 2 errores estándar (≈ 95%).

In [ ]:
def comparar(nombre: str, metrica: str = "pr_auc", base: str = "base") -> dict:
    """Diferencia contra la base con el error estandar de la diferencia."""
    a = ablaciones.loc[ablaciones["run_name"] == nombre, metrica]
    b = ablaciones.loc[ablaciones["run_name"] == base, metrica]
    delta = a.mean() - b.mean()
    ee = float(np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)))
    z = delta / ee if ee else np.nan
    if abs(z) < 2:
        veredicto = "indistinguible"
    else:
        veredicto = "mejora" if delta > 0 else "empeora"
    return {"configuracion": nombre, "PR-AUC": a.mean(), f"delta vs base": delta,
            "error estandar": ee, "z": z, "veredicto": veredicto}


comparaciones = pd.DataFrame([comparar(n) for n in ORDEN if n != "base"])
comparaciones = comparaciones.sort_values("delta vs base", ascending=False)
comparaciones

In [ ]:
GRUPOS = {
    "marcador": ["sin_marcador"],
    "módulos": ["solo_texto", "solo_tabular"],
    "fuga": ["con_cart"],
    "ratio d_ff": ["ratio_1", "ratio_2", "ratio_8"],
    "cabezas": ["cabezas_1", "cabezas_2", "cabezas_8"],
    "ancho": ["ancho_32", "d48_6cabezas", "ancho_96"],
    "profundidad": ["capas_2", "capas_4"],
    "vocabulario": ["vocab_256"],
}
GRUPO_DE = {n: g for g, nombres in GRUPOS.items() for n in nombres}

comp = comparaciones.set_index("configuracion")
orden_fig = [n for n in ORDEN if n != "base"]
delta = comp.loc[orden_fig, "delta vs base"]
error = comp.loc[orden_fig, "error estandar"]
color = [ROJO if abs(z) >= 2 and d < 0 else VERDE if abs(z) >= 2 else GRIS
         for d, z in zip(delta, comp.loc[orden_fig, "z"])]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(orden_fig, delta, xerr=2 * error, color=color, edgecolor="white", capsize=3)
ax.axvline(0, color="black", lw=1)
ax.invert_yaxis()
ax.set_xlabel("Δ PR-AUC contra la configuración base (barras de ±2 errores estándar)")
ax.set_title("Ablación: qué mueve la aguja y qué no", loc="left")
etiquetas = [f"{n}  ({GRUPO_DE[n]})" for n in orden_fig]
ax.set_yticks(range(len(orden_fig)), etiquetas)
fig.tight_layout()
guardar(fig, "fig_11_ablacion")

---
## 3. El hallazgo central: el marcador de reputación

In [ ]:
ORDEN_MARCADOR = ["base", "solo_texto", "solo_tabular", "sin_marcador"]
marcador = summarize(
    ablaciones.loc[ablaciones["run_name"].isin(ORDEN_MARCADOR)]
).set_index("run_name").reindex(ORDEN_MARCADOR)
prevalencia = float(ablaciones["prevalence"].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colores = [AZUL, "#4299e1", GRIS, ROJO]
for ax, metrica, piso, titulo in [
    (ax1, "roc_auc", 0.5, "ROC-AUC"),
    (ax2, "pr_auc", prevalencia, "PR-AUC"),
]:
    ax.bar(marcador.index, marcador[metrica], yerr=marcador[f"{metrica}_sd"],
           color=colores, edgecolor="white", capsize=4)
    ax.axhline(piso, color="black", ls="--", lw=1)
    ax.text(0.02, piso, f" piso = {piso:.3f}", fontsize=8, va="bottom",
            transform=ax.get_yaxis_transform())
    ax.set_ylabel(titulo)
    ax.set_ylim(0, 1.02)
    ax.tick_params(axis="x", rotation=20)
ax1.set_title("Sin el marcador de reputación no hay problema que resolver", loc="left")
ax2.set_title("PR-AUC: el piso es la prevalencia", loc="left")
fig.tight_layout()
guardar(fig, "fig_12_marcador")
marcador[["roc_auc", "roc_auc_sd", "pr_auc", "pr_auc_sd"]]

---
## 4. Arquitectura: las cuatro familias

La configuración base es **`d_model` 64, 4 cabezas, 1 capa y `d_ff` 256**, es decir
`d_ff / d_model = 4`: el ratio de *Attention is all you need*, que usa 512 → 2048 en el
modelo base y 1024 → 4096 en el big.

> **Corrección respecto de la grilla anterior.** Las primeras recetas de capacidad
> tenían `d_ff` clavado en 128, con lo cual el ratio iba de 4.0 en la receta chica a
> **1.33 en la grande**: justo al revés de lo que se quería mostrar. La receta "grande"
> escalaba capas y atención pero dejaba el feed-forward proporcionalmente más angosto,
> hasta el punto de que la atención pasaba a llevarse el 60% de los parámetros del
> bloque cuando lo habitual es que sea el FFN el que se lleva ~2/3. La conclusión "la
> capacidad no cambia nada" no era sostenible con el FFN sin variar, así que la grilla
> se rehizo entera con el ratio fijo en 4.

Ahora la arquitectura se explora en **cuatro familias**, cada una moviendo un solo eje:

| Familia | Qué varía | Qué se mantiene |
|---|---|---|
| **ratio** | `d_ff` ∈ {64, 128, 256, 512} | todo lo demás. Es la única que se sale del ratio 4, porque es la que lo estudia |
| **cabezas** | `n_heads` ∈ {1, 2, 4, 8} | `d_model` 64 y `d_ff` 256, así que las cuatro tienen **exactamente los mismos parámetros**. Es el diseño de la Table 3(A) del paper |
| **ancho** | `d_model` ∈ {32, 48, 64, 96} | `d_head` 16 y ratio 4, que es cómo escala el paper (mantiene `d_k` fijo y mueve `h` con `d_model`) |
| **profundidad** | `n_layers` ∈ {1, 2, 4} | `d_model` 64 y `d_ff` 256 |

`cabezas_1` da `d_head` = 64, exactamente la dimensión por cabeza que usa el paper.
`d48_6cabezas` es el único punto con `d_head` 8 y ancho intermedio.

Si las cuatro familias salen planas, la lectura es que el problema **no necesita
capacidad**: la señal es un marcador de 4 tokens y alcanza con leerlo.

In [ ]:
# Cuatro familias, un panel cada una. La base aparece en las cuatro porque es el
# punto que todas comparten (d64 · h4 · L1 · d_ff 256).
FAMILIAS = [
    ("ratio d_ff / d_model", "d_ff",
     ["ratio_1", "ratio_2", "base", "ratio_8"], [64, 128, 256, 512]),
    ("cabezas (parámetros constantes)", "n_heads",
     ["cabezas_1", "cabezas_2", "base", "cabezas_8"], [1, 2, 4, 8]),
    ("ancho (d_head 16, ratio 4)", "d_model",
     ["ancho_32", "d48_6cabezas", "base", "ancho_96"], [32, 48, 64, 96]),
    ("profundidad", "n_layers",
     ["base", "capas_2", "capas_4"], [1, 2, 4]),
]

res_arq = summarize(ablaciones).set_index("run_name")
pr_base = res_arq.loc["base", "pr_auc"]

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharey=True)
for ax, (titulo, eje, nombres, valores) in zip(axes.ravel(), FAMILIAS):
    sub = res_arq.reindex(nombres)
    ax.errorbar(range(len(nombres)), sub["pr_auc"], yerr=sub["pr_auc_sd"],
                fmt="o", color=AZUL, capsize=4, markersize=7, lw=1.2)
    ax.axhline(pr_base, color=GRIS, ls="--", lw=1)
    ax.set_xticks(range(len(nombres)), [str(v) for v in valores])
    ax.set_xlim(-0.4, len(nombres) - 0.6)
    ax.set_xlabel(eje)
    ax.set_title(titulo, loc="left", fontsize=10)
    # Se marca cuál de los puntos es la base.
    i_base = nombres.index("base")
    ax.scatter([i_base], [pr_base], s=140, facecolors="none", edgecolors=ROJO, lw=1.6, zorder=5)

for ax in axes[:, 0]:
    ax.set_ylabel("PR-AUC (media ± desvío, 25 corridas)")
fig.suptitle("Arquitectura: cuatro ejes, ratio d_ff/d_model = 4 salvo en el primero  "
             "(círculo rojo = base)", y=1.01, fontsize=11)
fig.tight_layout()
guardar(fig, "fig_13_arquitectura")
plt.show()

# La tabla, con el ratio explícito para que se vea la convención.
from src.ablations import GRILLA
geo = {c.name: (c.d_model, c.n_heads, c.n_layers, c.d_ff) for c, _ in GRILLA}
filas_arq = []
for _, _, nombres, _ in FAMILIAS:
    for n in nombres:
        dm, h, L, ff = geo[n]
        filas_arq.append({"config": n, "d_model": dm, "n_heads": h, "d_head": dm // h,
                          "n_layers": L, "d_ff": ff, "ratio": round(ff / dm, 2),
                          "PR-AUC": round(res_arq.loc[n, "pr_auc"], 4),
                          "sd": round(res_arq.loc[n, "pr_auc_sd"], 4)})
pd.DataFrame(filas_arq).drop_duplicates(subset="config").reset_index(drop=True)


---
## 5. Tokenizador: el vocabulario que fragmenta el marcador

El corpus tiene 622 tipos de palabra y el BPE satura en 1.355 tokens: pedir más no cambia
nada. Hacia abajo sí: con 256 tokens el marcador pasa de **4 sub-tokens a 7.26** y el modelo
tiene que recomponerlo antes de poder usarlo.

In [ ]:
vocabs = ["vocab_256", "base"]
etiquetas_vocab = ["256\n(marcador en 7.26 tokens)", "1.355 = satura\n(marcador en 4 tokens)"]
res_vocab = summarize(ablaciones.loc[ablaciones["run_name"].isin(vocabs)])
res_vocab = res_vocab.set_index("run_name").reindex(vocabs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(etiquetas_vocab, res_vocab["pr_auc"], yerr=res_vocab["pr_auc_sd"],
       color=[NARANJA, AZUL], edgecolor="white", capsize=5, width=0.5)
ax.set_ylabel("PR-AUC")
ax.set_title("Vocabulario BPE: qué cuesta fragmentar el marcador", loc="left")
fig.tight_layout()
guardar(fig, "fig_14_vocabulario")
res_vocab[["pr_auc", "pr_auc_sd", "roc_auc", "roc_auc_sd"]]

---
## 6. La fuga

`cart` está excluida por una razón **causal**, no métrica: su valor es posterior al momento
de la predicción. Esta fila mide qué habría pasado si se la hubiera dejado, y convierte la
trampa en un resultado presentable.

In [ ]:
pd.DataFrame([comparar("con_cart"), comparar("sin_marcador")])

---
# Fase 10 — Evaluación final

## 7. La configuración final y el test

La configuración se eligió **solo con la CV**, con una regla fijada de antemano: la de mayor
PR-AUC medio, salvo que su ventaja sobre la base no supere un error estándar, en cuyo caso
gana la base por simplicidad. Quedan descalificadas de entrada `con_cart` (usa una columna
posterior al momento de la predicción) y `sin_marcador` / `solo_tabular`, que son
ablaciones diagnósticas: miden qué pasa cuando se le saca al modelo lo que necesita.

Como reentrenar sobre `dev` completo deja al early stopping sin partición, la cantidad de
épocas se fijó en la mediana de la época óptima de la CV de esa misma configuración: es
información de `dev`, no de `test`.

In [ ]:
final = resultados.loc[resultados["split"] == "test"]
assert len(final) == 1, "el test se evalua UNA sola vez"
fila = final.iloc[0]

pd.DataFrame([
    {"metrica": "prevalencia del test", "valor": f"{fila['prevalence']:.4f}",
     "lectura": "el piso del PR-AUC"},
    {"metrica": "PR-AUC (average precision)", "valor": f"{fila['pr_auc']:.4f}",
     "lectura": f"lift {fila['pr_auc'] / fila['prevalence']:.2f}x sobre el azar"},
    {"metrica": "ROC-AUC", "valor": f"{fila['roc_auc']:.4f}", "lectura": ""},
    {"metrica": "log loss", "valor": f"{fila['log_loss']:.4f}", "lectura": ""},
    {"metrica": "Brier", "valor": f"{fila['brier']:.4f}", "lectura": ""},
    {"metrica": "configuracion", "valor": fila["run_name"], "lectura": fila["features"]},
])

In [ ]:
# Reconstruccion del modelo final para las secciones de interpretabilidad.
# Se usan las MISMAS predicciones que se reportaron: el test se toco una vez.
from dataclasses import replace

from src.data.load import GROUP, TARGET, load_raw
from src.data.splits import load_splits
from src.final_eval import (
    atencion_del_cls, atencion_por_token, bootstrap_ci, curva_calibracion,
    elegir_configuracion, map_por_query, metricas_de_ranking,
)
from src.train import prepare_fold, train_model

df, splits = load_raw(), load_splits()
config, _ = elegir_configuracion(verbose=False)
epocas = int(fila["epochs_ran"])
run = replace(config, name=f"{config.name}_final", epochs=epocas, patience=epocas + 1)

fold_final = prepare_fold(df, splits.dev, splits.test, run)
# select_best=False: se entrena la cantidad de epocas que fijo la CV y se reporta
# el estado final. Elegir la mejor epoca mirando la metrica de test seria
# espiarlo, aunque no lo parezca.
resultado_final = train_model(fold_final, run, select_best=False)
y_true, y_prob = resultado_final.y_true, resultado_final.y_prob
grupos_test = df[GROUP].to_numpy()[splits.test]

print(f"reproducido: PR-AUC {resultado_final.best['pr_auc']:.4f} "
      f"(reportado {fila['pr_auc']:.4f})")

In [ ]:
bootstrap_ci(y_true, y_prob, grupos_test)

Los intervalos se calculan remuestreando **queries**, no filas: las impresiones de una misma
búsqueda comparten filtros, contexto y el sorteo del marcador, así que tratarlas como
independientes daría intervalos artificialmente angostos.

---
## 8. Calibración

In [ ]:
calibracion, ece = curva_calibracion(y_true, y_prob)
print(f"ECE (error de calibracion esperado) = {ece:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2),
                               gridspec_kw={"width_ratios": [1, 1.1]})
ax1.plot([0, 1], [0, 1], ls="--", color="black", lw=1, label="calibración perfecta")
ax1.plot(calibracion["prob. media predicha"], calibracion["frecuencia observada"],
         "o-", color=AZUL, label="modelo")
ax1.set(xlabel="probabilidad media predicha", ylabel="frecuencia observada de compra")
ax1.set_title("Curva de calibración por decil", loc="left")
ax1.legend(fontsize=8)

ax2.hist(y_prob[y_true == 0], bins=40, alpha=0.7, color=GRIS, label="no comprados")
ax2.hist(y_prob[y_true == 1], bins=40, alpha=0.7, color=ROJO, label="comprados")
ax2.set(xlabel="BTR predicho", ylabel="impresiones", yscale="log")
ax2.set_title("Distribución del BTR predicho", loc="left")
ax2.legend(fontsize=8)
fig.tight_layout()
guardar(fig, "fig_15_calibracion")
calibracion

---
## 9. Curvas ROC y PR contra los baselines

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

from src import baselines as B
from src.evaluate import compute_metrics

# Los baselines se reentrenan sobre dev y se evaluan sobre el mismo test, para
# que la comparacion sea contra las mismas filas.
comparables = ["prior", "gbdt_tabular", "tfidf_texto", "gbdt_sufijo"]
curvas = {"transformer (final)": y_prob}
for nombre in comparables:
    baseline = next(b for b in B.BASELINES if b.nombre == nombre)
    prob, _ = baseline.fn(df.iloc[splits.dev], df.iloc[splits.test], B.SEED)
    curvas[nombre] = prob

pd.DataFrame([
    {"modelo": nombre, **{k: round(v, 4) for k, v in compute_metrics(y_true, prob).items()}}
    for nombre, prob in curvas.items()
])

In [ ]:
COLOR_MODELO = {"transformer (final)": AZUL, "gbdt_sufijo": ROJO,
                "tfidf_texto": VERDE, "gbdt_tabular": "#805ad5", "prior": GRIS}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.8))
for nombre, prob in curvas.items():
    m = compute_metrics(y_true, prob)
    ancho = 2.0 if nombre.startswith("transformer") else 1.4
    fpr, tpr, _ = roc_curve(y_true, prob)
    ax_roc.plot(fpr, tpr, color=COLOR_MODELO[nombre], lw=ancho,
                label=f"{nombre}  ({m['roc_auc']:.3f})")
    precision, recall, _ = precision_recall_curve(y_true, prob)
    ax_pr.plot(recall, precision, color=COLOR_MODELO[nombre], lw=ancho,
               label=f"{nombre}  ({m['pr_auc']:.3f})")

ax_roc.plot([0, 1], [0, 1], ls="--", color="black", lw=1)
ax_roc.set(xlabel="tasa de falsos positivos", ylabel="tasa de verdaderos positivos")
ax_roc.set_title("ROC sobre test", loc="left")
ax_roc.legend(loc="lower right", fontsize=8)

prev_test = float(y_true.mean())
ax_pr.axhline(prev_test, color="black", ls="--", lw=1)
ax_pr.text(0.02, prev_test + 0.02, f"prevalencia = {prev_test:.4f}", fontsize=8)
ax_pr.set(xlabel="recall", ylabel="precision", ylim=(0, 1.02))
ax_pr.set_title("Precision-Recall sobre test", loc="left")
ax_pr.legend(loc="upper right", fontsize=8)
fig.tight_layout()
guardar(fig, "fig_16_curvas_final")

---
## 10. Diagnóstico de over/underfitting

In [ ]:
# Curvas de entrenamiento del modelo base (Fase 8), sobre train/valid.
curva = pd.read_csv(ROOT / "experiments" / "curves" / "base.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(curva["epoca"], curva["train_loss"], color=AZUL, label="loss de train")
ax1.plot(curva["epoca"], curva["log_loss"], color=NARANJA, label="loss de validación")
ax1.set(xlabel="época", ylabel="binary cross-entropy")
ax1.set_title("Train contra validación", loc="left")
ax1.legend(fontsize=8)

ax2.plot(curva["epoca"], curva["pr_auc"], color=AZUL, label="PR-AUC")
ax2.plot(curva["epoca"], curva["roc_auc"], color=VERDE, label="ROC-AUC")
mejor = curva.loc[curva["pr_auc"].idxmax()]
ax2.axvline(mejor["epoca"], color=ROJO, ls="--", lw=1,
            label=f"mejor época ({int(mejor['epoca'])})")
ax2.set(xlabel="época", ylabel="métrica de validación")
ax2.set_title("Métricas de validación por época", loc="left")
ax2.legend(fontsize=8)
fig.tight_layout()
guardar(fig, "fig_17_curvas_entrenamiento")

In [ ]:
# Learning curve: performance contra la fraccion de train usada. Se mide sobre
# valid (dentro de dev), nunca sobre test.
from src.utils.seed import set_seed

FRACCIONES = [0.1, 0.25, 0.5, 0.75, 1.0]
filas_lc = []
for fraccion in FRACCIONES:
    n = int(len(splits.train) * fraccion)
    rng = np.random.default_rng(42)
    idx = np.sort(rng.choice(splits.train, size=n, replace=False))
    run_lc = replace(config, name=f"lc_{fraccion}")
    res = train_model(prepare_fold(df, idx, splits.valid, run_lc), run_lc)
    filas_lc.append({"fraccion": fraccion, "n_train": n,
                     "pr_auc": res.best["pr_auc"], "roc_auc": res.best["roc_auc"]})
learning_curve = pd.DataFrame(filas_lc)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(learning_curve["n_train"], learning_curve["pr_auc"], "o-", color=AZUL,
        label="PR-AUC de validación")
ax.set(xlabel="filas de entrenamiento", ylabel="PR-AUC")
ax.set_title("Learning curve: ¿el límite son los datos o la arquitectura?", loc="left")
ax.legend(fontsize=8)
fig.tight_layout()
guardar(fig, "fig_18_learning_curve")
learning_curve

---
## 11. Mapas de atención

La hipótesis a verificar: **el modelo concentra la atención del token `[CLS]` sobre los
tokens del marcador de reputación**. Es la evidencia visual de que el mecanismo de atención
aprendió a localizar la señal que el EDA encontró, y conecta la arquitectura con el
hallazgo.

Se promedia sobre **todo el test**, no sobre una muestra elegida a mano.

In [ ]:
pesos, mascara = atencion_del_cls(resultado_final.model, fold_final)
tokens = [fold_final.tokenizer.encode_one(a, b)
          for a, b in zip(fold_final.pipeline.transform(df.iloc[splits.test]).text_a,
                          fold_final.pipeline.transform(df.iloc[splits.test]).text_b)]

top_tokens = atencion_por_token(pesos, mascara, tokens, top=20)
top_tokens

El nivel de reputación está codificado **en dos lugares**: el sufijo entre paréntesis del
título y la última oración de la descripción. Así que la atención se reparte en tres zonas,
no en dos, y la pregunta es si el modelo va a las dos codificaciones o solo a una.

In [ ]:
from src.data.features import extract_reputation_sentence
from src.data.load import SUFFIX_NONE, extract_suffix

test_df = df.iloc[splits.test]
sufijos = extract_suffix(test_df["title"]).to_numpy()
oraciones = extract_reputation_sentence(test_df["description"]).to_numpy()

# Vocabulario de cada zona (el pre-tokenizador corta por espacios y puntuacion).
palabras_sufijo = {p for s in np.unique(sufijos) if s != SUFFIX_NONE
                   for p in s.replace("#", "").split()}
palabras_oracion = {p.strip(".,") for o in np.unique(oraciones) if o
                    for p in o.split()} - palabras_sufijo

def zona(token: str) -> int:
    if token in palabras_sufijo:
        return 0
    return 1 if token in palabras_oracion else 2


ZONAS = ["sufijo del título", "oración de reputación", "resto del texto"]
atencion_zona = np.zeros((len(tokens), 3))
tokens_zona = np.zeros((len(tokens), 3))
for i, (fila_pesos, fila_mascara, fila_tokens) in enumerate(zip(pesos, mascara, tokens)):
    for j, token in enumerate(fila_tokens):
        if j < len(fila_mascara) and fila_mascara[j]:
            atencion_zona[i, zona(token)] += float(fila_pesos[j])
            tokens_zona[i, zona(token)] += 1

resumen_atencion = pd.DataFrame({
    "zona": ZONAS,
    "tokens (media)": tokens_zona.mean(axis=0),
    "% de los tokens": 100 * tokens_zona.mean(axis=0) / tokens_zona.mean(axis=0).sum(),
    "% de la atención del [CLS]": 100 * atencion_zona.mean(axis=0),
})
resumen_atencion["concentración (atención / tokens)"] = (
    resumen_atencion["% de la atención del [CLS]"] / resumen_atencion["% de los tokens"]
)
resumen_atencion

In [ ]:
COLOR_ZONA = [ROJO, NARANJA, GRIS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5),
                               gridspec_kw={"width_ratios": [1.3, 1]})

top = top_tokens.head(15).iloc[::-1]
ax1.barh(top["token"], top["atencion media"],
         color=[COLOR_ZONA[zona(t)] for t in top["token"]], edgecolor="white")
ax1.set_xlabel("atención media del [CLS] (promedio sobre el test)")
ax1.set_title("A qué tokens mira el modelo", loc="left")
manijas = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOR_ZONA]
ax1.legend(manijas, ZONAS, fontsize=8, loc="lower right")

x = np.arange(3)
ax2.bar(x - 0.2, resumen_atencion["% de los tokens"], 0.4, color=GRIS,
        label="% de los tokens")
ax2.bar(x + 0.2, resumen_atencion["% de la atención del [CLS]"], 0.4, color=ROJO,
        label="% de la atención")
ax2.set_xticks(x, [z.replace(" ", "\n") for z in ZONAS])
ax2.set_ylabel("porcentaje")
ax2.set_title("La atención se concentra donde está la señal", loc="left")
ax2.legend(fontsize=8)
fig.tight_layout()
guardar(fig, "fig_19_atencion")

In [ ]:
# Un ejemplo concreto: el mapa completo de una impresion de nivel ALTO.
from src.data.load import extract_suffix

idx_alto = int(np.flatnonzero(np.isin(sufijos, ["Best Seller", "Top Rated",
                                                "#1 Pick", "Customer Favorite"]))[0])
tokens_ej = tokens[idx_alto]
validos = int(mascara[idx_alto].sum())

fig, ax = plt.subplots(figsize=(13, 2.6))
ax.imshow(pesos[idx_alto][:validos][None, :], aspect="auto", cmap="Reds")
ax.set_yticks([])
ax.set_xticks(range(validos), tokens_ej[:validos], rotation=90, fontsize=7)
ax.set_title(f"Atención del [CLS] en una impresión de nivel ALTO "
             f"({sufijos[idx_alto]})", loc="left")
ax.grid(False)
fig.tight_layout()
guardar(fig, "fig_20_atencion_ejemplo")

### Contraste con la importancia de features del GBDT

Si el Transformer aprendió a leer el marcador, el GBDT —que lo recibe servido como
categórica— tendría que apoyarse en la misma señal. Se mide con *permutation importance*
sobre el test: cuánto cae el PR-AUC al permutar cada columna.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, average_precision_score

from src.data.features import CATEGORICAS, CATEGORICAS_OPCIONALES, FeaturePipeline
from src.baselines import _hist_gbdt

pipe_gbdt = FeaturePipeline(categoricas=CATEGORICAS + CATEGORICAS_OPCIONALES)
pipe_gbdt.fit(df.iloc[splits.dev])
Xdev, Xtest = pipe_gbdt.transform(df.iloc[splits.dev]), pipe_gbdt.transform(df.iloc[splits.test])
gbdt = _hist_gbdt(Xdev, seed=42)
gbdt.fit(np.hstack([Xdev.num, Xdev.cat]), Xdev.y)

nombres = list(Xdev.nombres_num) + list(Xdev.nombres_cat)
importancia = permutation_importance(
    gbdt, np.hstack([Xtest.num, Xtest.cat]), Xtest.y, n_repeats=5, random_state=42,
    scoring=make_scorer(average_precision_score, response_method="predict_proba"),
)
tabla_gbdt = pd.DataFrame({
    "feature": nombres,
    "caída de PR-AUC al permutar": importancia.importances_mean,
    "desvío": importancia.importances_std,
}).sort_values("caída de PR-AUC al permutar", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 4))
top_gbdt = tabla_gbdt.iloc[::-1]
ax.barh(top_gbdt["feature"], top_gbdt["caída de PR-AUC al permutar"],
        xerr=top_gbdt["desvío"],
        color=[ROJO if f == "sufijo" else GRIS for f in top_gbdt["feature"]],
        edgecolor="white", capsize=3)
ax.set_xlabel("caída de PR-AUC al permutar la columna (test)")
ax.set_title("Importancia por permutación del GBDT: la misma señal", loc="left")
fig.tight_layout()
guardar(fig, "fig_21_importancia_gbdt")
tabla_gbdt

---
## 12. Métricas de ranking

In [ ]:
ranking = metricas_de_ranking(y_true, y_prob, grupos_test)
print(f"mAP por query = {map_por_query(y_true, y_prob, grupos_test):.4f}")
ranking

`precision@k` por query es la lectura del caso de uso: si hay que promocionar k productos
dentro de una búsqueda, cuántos de ellos terminan comprados. El `mAP` por query mide la
calidad del orden *dentro* de cada búsqueda, que es donde el modelo se usaría.